In [0]:
import json
import os
from urllib.parse import urljoin

import pandas as pd
import requests

BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
# 1.Позив API-ја. Написати функцију get_json(url: str) -> dict која шаље GET захтев.
# Подиже грешку ако статус није успешан и враћа одговор као речник ако јесте.
# get_json(BASE_URL + 'status.json')['service'] враћа 'Northern Star ShopAPI'
# get_json(BASE_URL + 'orders/page-99.json') подиже requests.HTTPError

def get_json(url: str) -> dict:
    r = requests.get(url, timeout=10)
    r.raise_for_status()
    data = r.json()
    return data


In [0]:
# 2. Странице. Написати функцију fetch_all(first_url: str) -> list која прати next линкове и враћа записе са свих страна у једној листи Помоћ: urljoin(url, page['next'])
# Input: BASE_URL + 'orders/page-1.json' даје 1852 записа

def fetch_all(first_url: str) -> list:
    records = []
    url = first_url
    while url:
        page = get_json(url)
        records.extend(page['data'])
        url = urljoin(url, page['next']) if page['next'] else None
    return records
    
# fetch_all(BASE_URL + 'orders/page-1.json')

In [0]:
# 3. Чување сирових података. Написати функцију save_json(records: list, path: str) -> None која листу речника уписује у JSON фајл без икаквих измена.
#     json.dump(records, f, ensure_ascii=False, indent=2)
#     save_json(orders, os.path.join(RAW_DIR, 'orders.json')) -> прави се фајл data/raw/orders.json

def save_json(records: list, path: str) -> None:
    
    with open(path, 'w') as f:
        json.dump(records, f, indent=2, ensure_ascii=False)

# orders = fetch_all(BASE_URL + 'orders/page-1.json')
# save_json(orders, os.path.join(RAW_DIR, 'orders.json'))

In [0]:
# 4. Преузимање фајла. Функција је дата: копирати је без измена у своју ћелију и користити у задатку 5

def download_file(url: str, path: str) -> int:
#Download a file and save it unchanged. Returns its size in bytes."""
    
    response = requests.get(url, timeout=30) 
    response.raise_for_status() # 404 and other errors stop here
    
    with open(path,'wb') as f:
        # 'wb' = write bytes, needed for Parquet
        f.write(response.content)

    # .content = the file's bytes, unchanged
    return len(response.content)

# download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR,'teams.csv'))

In [0]:
# 5. Написати функцију extract(raw_dir: str) -> dict која преузима сва четири извора у raw_dir
# и враћа речник са четири DataFrame-а: 'orders', 'products', 'teams','matches'

#     1. Поруџбине читамо помоћу API-ја (fetch_all) (orders.json)
#     2. Фајлови products.csv,teams.csv,matches.parquet се добијају помоћу функције download_file
#     3. Поруџбине у DataFrame: pd.json_normalize(orders, sep='_')
#         Након тога преименовати колоне customer_name у customer, customer_city у city
#     4. Остало читамо из raw_dir (где смо у кораку 2 скинули податке)
#         По потреби,pd.read_csv(...) или pd.read_parquet(...)
#     5. Провера: orders (1852, 8), products (108, 7), teams (20, 5), matches (120, 9)

def extract(raw_dir: str) -> dict:
    orders = fetch_all(BASE_URL + 'orders/page-1.json')
    orders_df = pd.json_normalize(orders, sep='_')
    orders_df = orders_df.rename(columns={'customer_name': 'customer', 'customer_city': 'city'})
    download_file(FILES_URL + 'products.csv', os.path.join(RAW_DIR,'products.csv'))
    products_df = pd.read_csv(os.path.join(RAW_DIR,'products.csv'))
    download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR,'teams.csv'))
    teams_df = pd.read_csv(os.path.join(RAW_DIR,'teams.csv'))
    download_file(FILES_URL + 'matches.parquet', os.path.join(RAW_DIR,'matches.parquet'))
    matches_df = pd.read_parquet(os.path.join(RAW_DIR,'matches.parquet'))

    result = {
        'orders': orders_df,
        'products': products_df,
        'teams': teams_df,
        'matches': matches_df
    }
    return result

#extract(RAW_DIR)


In [0]:
# 6. Производи. Написати функцију clean_products(df: pd.DataFrame) -> pd.DataFrame која чисти каталог производа.

#     Ако се понавља исти производ, садржати први ред за сваки product_id
#     team_code треба да буде без размака, има велика слова (npm -> NPM), празан ред остаје празан (то су производи лиге, без тима)
#     category треба да буде без размака, има мала слова
#     price је текст у три облика '1490.00', '1490 RSD', '1490,00' а треба да постане само float (у сваком случају)
#         Помоћ: .str.replace(' RSD', '', regex=False),pd.to_numeric(...)
#     active трансформишемо у True/False('yes', 'Yes', 'YES' постају True), све остало постаје False
#     Output: 105 редова, price је типа float64, active је True за 101 производ
#         Ред 1003 изгледа овако: {'team_code': 'NPM', 'category': 'scarves','price': 1490.0, 'active': True, ...}

def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    df_product_clean = df.copy()
    df_product_clean = df_product_clean.drop_duplicates('product_id')
    df_product_clean['team_code']= df_product_clean['team_code'].str.strip().str.upper()
    df_product_clean['category'] = df_product_clean['category'].str.strip().str.lower()
    df_product_clean['price'] = df_product_clean['price'].str.replace(' RSD', '').str.replace(',', '.')
    df_product_clean['price'] = pd.to_numeric(df_product_clean['price']).astype(float)
    df_product_clean['active'] = df_product_clean['active'].str.lower().map({'yes': True}).fillna(False)

    return df_product_clean
    

# df = extract(RAW_DIR)['products']
# clean_products(df)

In [0]:
# 7. Поруџбине. Написати функцију clean_orders(df: pd.DataFrame) -> pd.DataFrame 
#     order_id, customer, city, payment без размака;
#     order_id велика слова
#     customer и city -> title()
#     Град који недостаје (isna) -> 'Unknown'

def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    df_orders_clean = df.copy()
    for col in ['order_id', 'customer', 'city', 'payment']:
        df_orders_clean[col] = df_orders_clean[col].str.strip()
    df_orders_clean ['order_id'] = df_orders_clean ['order_id'].str.upper()
    for col in ['customer', 'city']:
        df_orders_clean[col] = df_orders_clean[col].str.title()
    df_orders_clean['city'] = df_orders_clean['city'].fillna('Unknown')

    return df_orders_clean
    

# df = extract(RAW_DIR)['orders']
# clean_orders(df)

In [0]:
# 8. Дупликати. Написати функцију remove_duplicate_orders(df: pd.DataFrame) -> tuple која враћа
# (unique, duplicates) по order_id, задржава први
# Output: 1840 јединствених, 12 дупликата

def remove_duplicate_orders(df: pd.DataFrame) -> tuple:
    is_dup = df.duplicated('order_id', keep='first')
    return df[~is_dup], df[is_dup]

# df = extract(RAW_DIR)['orders']
# remove_duplicate_orders(df)

In [0]:
# 9. Написати функцију validate_order(row: dict, product_ids: set) -> str која враћа 'OK' за исправан ред, у супротном разлог 
#     Провере радити редом и вратити први пронађени проблем
#         order_id не почиње са NSS- -> 'invalid_order'
#         quantity недостаје -> 'missing_quantity'
#         quantity није цео број већи од нуле -> 'invalid_quantity'
#         unit_price није број већи од нуле -> 'invalid_price'
#         product_id не постоји међу производима -> 'unknown_product'
# Помоћ: 
#   pd.to_numeric(value, errors='coerce') претвара '2' у 2, а 'abc' у NaN
#   pd.isna(value) препознаје и None и NaN

def validate_order(row: dict, product_ids: set) -> str:
    quantity = pd.to_numeric(row['quantity'], errors='coerce')
    price = pd.to_numeric(row['unit_price'], errors='coerce')
    
    if not row['order_id'].startswith('NSS-'):
        return 'invalid_order'
    if pd.isna(row['quantity']):
        return 'missing_quantity'
    if pd.isna(quantity) or quantity <= 0 or quantity != int(quantity):
        return 'invalid_quantity'
    if pd.isna(price) or price <= 0:
        return 'invalid_price'
    if row['product_id'] not in product_ids:
        return 'unknown_product'
    return 'OK'
    

In [0]:
# 10. Написати функцију transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame
# Типови: 
#     quantity -> int  
#     unit_price -> float
#     order_date -> датум (pd.to_datetime)
# Нова колона total(quantity * unit_price)
# Из очишћених производа (products) додати product_name, team_code и category (merge по product_id)
# Output: 1800 редова и 12 колона (8 из поруџбина, total, и 3 из производа)

def transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['quantity'] = pd.to_numeric(df['quantity']).astype(int)
    df['unit_price'] = pd.to_numeric(df['unit_price']).astype(float)
    df['order_date'] = pd.to_datetime(df['order_date'])
    df['total'] = df['quantity'] * df['unit_price']
    product_slice = products[['product_id', 'product_name', 'team_code', 'category']]
    return df.merge(product_slice, on='product_id', how='left')

In [0]:
# 11. Написати функцију split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple која сирове поруџбине чисти, избацује дупликате, проверава и враћа торку (valid,rejected)
# valid су трансформисани исправни редови
# rejected су очишћени неисправни редови и дупликати, са колоном reason (дупликати за разлог треба да имају
# 'duplicate')
# Output: 1800 исправних, 52 одбијених (од тога 12 дупликата)

# def split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple:
#     df = clean_orders(df)
#     unique, duplicates = remove_duplicate_orders(df)



In [0]:
def load(outputs: dict, processed_dir:str) -> dict: 
    written = {}
    for name, df in outputs.items(): path = os.path.join(processed_dir, name)
    if name.endswith('.parquet'): 
        df.to_parquet(path, index=False)
    else: 
        df.to_csv(path, index=False) 
    written[name] = len(df)
    return written

In [0]:
def run_pipeline(raw_dir: str, processed_dir:str) -> dict: 
    data = extract(raw_dir) 
    products = clean_products(data['products']) 
    # valid, rejected = split_orders(data['orders'], products) 
#     results = team_results(data['matches'], data['teams']) 
#     windows = match_windows(results) 
#     outputs = {
#         'clean_orders.parquet': valid, 
#         'rejected_orders.csv': rejected,
#         'products.csv': products,
#         'league_table.csv': league_table(results),
#         'revenue_by_team.csv': revenue_by(valid.dropna(subset=['team_code']),'team_code'),
#         'revenue_by_category.csv': revenue_by(valid,'category'),
#         'sales_after_results.csv': sales_after_results(valid, windows), 
#         } 
#     written = load(outputs, processed_dir) 
#     summary = {
#         'total_rows': len(data['orders']),
#         'duplicates': int((rejected['reason'] == 'duplicate').sum()),
#         'valid_rows': len(valid),
#         'rejected_rows': len(rejected),
#         'total_revenue': round(float(valid['total'].sum()),2),
#         'files_written': len(written) + 1, 
#         } 
#     pd.DataFrame(list(summary.items()), columns=['metric', 'value']).to_csv( os.path.join(processed_dir,
# 'summary.csv'), index=False)
    
#     return summary
    return products

run_pipeline(RAW_DIR, PROCESSED_DIR)

/home/spark-70abb116-bcec-4692-b49a-08/.ipykernel/75/command-8074213805319516-2731955650:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_product_clean['active'] = df_product_clean['active'].str.lower().map({'yes': True}).fillna(False)


,product_id,sku,product_name,team_code,category,price,active
0,1001,NPM-JER-1001,NPM Home Jersey,NPM,jerseys,7490.0,True
1,1002,NPM-JER-1002,NPM Away Jersey,NPM,jerseys,6990.0,True
2,1003,NPM-SCA-1003,NPM Scarf,NPM,scarves,1490.0,True
3,1004,NPM-CAP-1004,NPM Cap,NPM,caps,2290.0,True
4,1005,NPM-ACC-1005,NPM Mug,NPM,accessories,990.0,True
...,...,...,...,...,...,...,...
100,1101,NSL-BAL-1101,NSL Match Ball,NaN,balls,7490.0,True
101,1102,NSL-BAL-1102,NSL Training Ball,NaN,balls,2990.0,True
102,1103,NSL-ACC-1103,NSL Water Bottle,NaN,accessories,890.0,True
103,1104,NSL-ACC-1104,NSL Backpack,NaN,accessories,3990.0,True
